<a href="https://colab.research.google.com/github/Ubirajarajubatusfan/CHO-oxidise-stress/blob/main/mapping_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNA-seq пайплайн для 6 образцов

## 📋 Описание

Скрипт для полного анализа RNA-seq данных 6 образцов китайского хомячка (*Cricetulus griseus*):

- **FastQC** — контроль качества ридов
- **MultiQC** — агрегация отчётов качества
- **HISAT2** — выравнивание на референсный геном
- **featureCounts** — подсчёт экспрессии генов

---

## 📦 Список SRA образцов

| № | SRA ID |
|---|--------|
| 1 | SRR22246869 |
| 2 | SRR22246870 |
| 3 | SRR22246871 |
| 4 | SRR22246901 |
| 5 | SRR22246900 |
| 6 | SRR22246899 |

---

## 🚀 Полный скрипт

Скопируй и сохрани в файл `rnaseq_pipeline.sh`:

```bash
#!/bin/bash
# RNA-seq пайплайн для 6 образцов (без Git)
# FastQC + MultiQC + HISAT2 + featureCounts

set -e  # Остановка при любой ошибке

echo "========================================="
echo "  RNA-seq анализ 6 образцов"
echo "========================================="

# Список SRA идентификаторов
SRA_IDS=(
    "SRR22246869"
    "SRR22246870"
    "SRR22246871"
    "SRR22246901"
    "SRR22246900"
    "SRR22246899"
)

# ============================================
# ЧАСТЬ 1: Обновление системы и установка базовых пакетов
# ============================================
echo "[1/8] Обновление системы и установка базовых пакетов..."

sudo apt update
sudo apt upgrade -y
sudo apt install -y build-essential curl wget vim nano

# ============================================
# ЧАСТЬ 2: Установка Miniconda
# ============================================
echo "[2/8] Установка Miniconda..."

if [ ! -d "$HOME/miniconda3" ]; then
    wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p $HOME/miniconda3
    rm /tmp/miniconda.sh
    echo "Miniconda установлен"
else
    echo "Miniconda уже установлен"
fi

# Инициализация Conda
eval "$($HOME/miniconda3/bin/conda shell.bash hook)"
conda init bash
source ~/.bashrc

# Проверка установки
conda --version

# ============================================
# ЧАСТЬ 3: Настройка каналов Conda
# ============================================
echo "[3/8] Настройка каналов Conda..."

conda config --add channels conda-forge
conda config --add channels bioconda
conda config --set channel_priority strict

# ============================================
# ЧАСТЬ 4: Создание Conda-окружений
# ============================================
echo "[4/8] Создание Conda-окружений..."

# Окружение для FastQC, MultiQC и SRA Toolkit
conda create -n fastqc_analysis -c bioconda -c conda-forge fastqc multiqc sra-tools -y

# Окружение для HISAT2 и featureCounts
conda create -n hisat_analysis -c bioconda -c conda-forge hisat2 subread samtools -y

echo "Окружения созданы"

# ============================================
# ЧАСТЬ 5: Создание структуры проекта
# ============================================
echo "[5/8] Создание структуры проекта..."

mkdir -p ~/Project/{reference,SRRfiles,fastqc-raw,fastqc_res,aligned,multiqc_res,logs,fastqRNA}

cd ~/Project

# Создание файла со списком SRA ID
printf "%s\n" "${SRA_IDS[@]}" > ~/Project/SRRfiles/SRR_Acc_List.txt
echo "Создан файл с SRA ID:"
cat ~/Project/SRRfiles/SRR_Acc_List.txt

# ============================================
# ЧАСТЬ 6: Загрузка референсных данных
# ============================================
echo "[6/8] Загрузка референсного генома и аннотации..."

cd ~/Project/reference

# Загрузка генома
if [ ! -f "GCF_000223135.1_CriGri_1.0_genomic.fna.gz" ]; then
    wget -P ~/Project/reference "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/223/135/GCF_000223135.1_CriGri_1.0/GCF_000223135.1_CriGri_1.0_genomic.fna.gz"
fi

# Загрузка аннотации
if [ ! -f "GCF_000223135.1_CriGri_1.0_genomic.gtf.gz" ]; then
    wget -P ~/Project/reference "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/223/135/GCF_000223135.1_CriGri_1.0/GCF_000223135.1_CriGri_1.0_genomic.gtf.gz"
fi

# Распаковка (для HISAT2 нужны несжатые файлы)
echo "Распаковка референсных файлов..."
gunzip -k -f *.fna.gz
gunzip -k -f *.gtf.gz

# Переименование аннотации для удобства
cp GCF_000223135.1_CriGri_1.0_genomic.gtf annotation.gtf

echo "Референсные файлы готовы"

# ============================================
# ЧАСТЬ 7: Загрузка SRA данных и FastQC
# ============================================
echo "[7/8] Загрузка SRA данных и контроль качества..."

conda activate fastqc_analysis

# Загрузка всех образцов
echo "Загрузка SRA данных для ${#SRA_IDS[@]} образцов..."

for sra_id in "${SRA_IDS[@]}"; do
    echo "Загрузка: $sra_id"
    prefetch "$sra_id" --max-size 50G || echo "Ошибка при загрузке $sra_id"
done

# Конвертация всех образцов в FASTQ (в fastqc-raw)
echo "Конвертация SRA в FASTQ..."
for sra_id in "${SRA_IDS[@]}"; do
    echo "Конвертация: $sra_id"
    fastq-dump "$sra_id" --gzip --split-3 --outdir ~/Project/fastqc-raw
done

# Копирование FASTQ файлов в fastqRNA (для HISAT2)
echo "Копирование файлов в fastqRNA..."
cp ~/Project/fastqc-raw/*.fastq.gz ~/Project/fastqRNA/

echo "Содержимое fastqRNA:"
ls -la ~/Project/fastqRNA/

# Запуск FastQC для всех файлов в fastqc-raw
echo "Запуск FastQC для всех образцов..."
fastqc ~/Project/fastqc-raw/*.fastq.gz -o ~/Project/fastqc_res -t 2

# Запуск MultiQC для всех отчётов
echo "Запуск MultiQC..."
multiqc -o ~/Project/multiqc_res ~/Project/fastqc_res/

echo "FastQC и MultiQC завершены"

# ============================================
# ЧАСТЬ 8: Выравнивание с HISAT2
# ============================================
echo "[8/8] Выравнивание ридов с HISAT2..."

conda activate hisat_analysis

cd ~/Project

# Создание директории для индекса
mkdir -p reference/index

# Построение индекса HISAT2
if [ ! -f "reference/index/genome.1.ht2" ]; then
    echo "Построение индекса HISAT2..."
    hisat2-build -p 2 reference/GCF_000223135.1_CriGri_1.0_genomic.fna reference/index/genome
fi

# Выравнивание для каждого образца с правильным параметром strandness
for sra_id in "${SRA_IDS[@]}"; do
    echo ""
    echo "========================================="
    echo "Выравнивание образца: $sra_id"
    echo "========================================="
    
    # Проверяем наличие файлов
    R1=~/Project/fastqRNA/${sra_id}_1.fastq.gz
    R2=~/Project/fastqRNA/${sra_id}_2.fastq.gz
    
    if [ -f "$R1" ] && [ -f "$R2" ]; then
        echo "Найдены файлы:"
        echo "  R1: $R1"
        echo "  R2: $R2"
        
        # Выравнивание с параметром RF (для stranded RNA-seq)
        echo "Запуск HISAT2..."
        hisat2 -p 2 --rna-strandness RF \
            -x reference/index/genome \
            -1 "$R1" \
            -2 "$R2" \
            2> "logs/${sra_id}_hisat2.log" | \
            samtools view -bS - > "aligned/${sra_id}.unsorted.bam"
        
        echo "Образец $sra_id выровнен"
    else
        echo "ПРЕДУПРЕЖДЕНИЕ: Файлы для $sra_id не найдены"
        echo "Ищем: $R1 и $R2"
    fi
done

# Обработка BAM файлов: сортировка, индексация, статистика
echo ""
echo "========================================="
echo "Обработка BAM файлов"
echo "========================================="

for sra_id in "${SRA_IDS[@]}"; do
    UNSORTED_BAM="aligned/${sra_id}.unsorted.bam"
    SORTED_BAM="aligned/${sra_id}.sorted.bam"
    
    if [ -f "$UNSORTED_BAM" ]; then
        echo "Обработка: $sra_id"
        
        # 1. Сортировка
        echo "  Сортировка BAM..."
        samtools sort -@ 2 "$UNSORTED_BAM" -o "$SORTED_BAM"
        
        # 2. Индексация
        echo "  Индексация BAM..."
        samtools index "$SORTED_BAM"
        
        # 3. Статистика выравнивания
        echo "  Статистика выравнивания..."
        samtools flagstat "$SORTED_BAM" > "aligned/${sra_id}_flagstat.txt"
        
        # Удаление несортированного BAM для экономии места
        rm "$UNSORTED_BAM"
        
        echo "  $sra_id готов"
    else
        echo "ПРЕДУПРЕЖДЕНИЕ: $UNSORTED_BAM не найден"
    fi
done

# Подсчёт экспрессии с featureCounts
echo ""
echo "========================================="
echo "Подсчёт экспрессии генов (featureCounts)"
echo "========================================="

# Собираем все сортированные BAM файлы
BAM_FILES=$(ls aligned/*.sorted.bam 2>/dev/null | tr '\n' ' ')

if [ ! -z "$BAM_FILES" ]; then
    featureCounts -p --countReadPairs \
        -t exon -g gene_id \
        -a reference/annotation.gtf \
        -o aligned/counts.txt \
        $BAM_FILES
    
    echo "Таблица экспрессии создана: aligned/counts.txt"
else
    echo "ОШИБКА: Не найдено BAM файлов для подсчёта"
fi

# Создание сводного отчёта о выравнивании
echo ""
echo "========================================="
echo "Создание сводного отчёта"
echo "========================================="

echo "Sample|Total_reads|Aligned_reads|Alignment_rate" > aligned/alignment_summary.txt
echo "-------|-----------|--------------|---------------" >> aligned/alignment_summary.txt

for sra_id in "${SRA_IDS[@]}"; do
    FLAGSTAT="aligned/${sra_id}_flagstat.txt"
    if [ -f "$FLAGSTAT" ]; then
        TOTAL=$(grep "total" "$FLAGSTAT" | head -1 | awk '{print $1}')
        ALIGNED=$(grep "aligned 0 times" "$FLAGSTAT" | awk '{print $5}')
        if [ -n "$TOTAL" ] && [ -n "$ALIGNED" ] && [ "$TOTAL" -gt 0 ]; then
            RATE=$(echo "scale=2; ($TOTAL - $ALIGNED) * 100 / $TOTAL" | bc)
            echo "$sra_id|$TOTAL|$((TOTAL - ALIGNED))|$RATE%" >> aligned/alignment_summary.txt
        else
            echo "$sra_id|N/A|N/A|N/A" >> aligned/alignment_summary.txt
        fi
    else
        echo "$sra_id|N/A|N/A|N/A" >> aligned/alignment_summary.txt
    fi
done

# ============================================
# ИТОГОВАЯ СТАТИСТИКА
# ============================================
echo ""
echo "========================================="
echo "        АНАЛИЗ УСПЕШНО ЗАВЕРШЁН!"
echo "========================================="
echo ""
echo "📊 РЕЗУЛЬТАТЫ:"
echo ""
echo "1. FastQC отчёты:"
echo "   ~/Project/fastqc_res/"
echo ""
echo "2. MultiQC сводный отчёт:"
echo "   ~/Project/multiqc_res/multiqc_report.html"
echo ""
echo "3. BAM файлы (выравнивания):"
for sra_id in "${SRA_IDS[@]}"; do
    if [ -f "aligned/${sra_id}.sorted.bam" ]; then
        echo "   ✓ ${sra_id}.sorted.bam"
    else
        echo "   ✗ ${sra_id}.sorted.bam"
    fi
done
echo ""
echo "4. Таблица экспрессии:"
echo "   ~/Project/aligned/counts.txt"
echo ""
echo "5. Статистика выравнивания:"
echo "   ~/Project/aligned/alignment_summary.txt"
echo ""
echo "========================================="
echo "Для просмотра результатов:"
echo "  cat ~/Project/aligned/alignment_summary.txt"
echo "  xdg-open ~/Project/multiqc_res/multiqc_report.html"
echo "========================================="